# 23 — Throughput Objective

**Engineering statement:** Verification allocation specifies decoding throughput.

Notebook 23 turns the first four notebooks into one systems objective. Notebook 00 introduced confidence-scheduled decoding. Notebook 07 framed verification as a scarce engineering resource. Notebook 13 showed how confidence schedules that resource. Notebook 17 showed how semi-autoregressive drafting specifies better confidence profiles. This notebook asks the next engineering question:

> **Which verification schedule maximizes end-to-end useful throughput?**


## Repository roadmap

```text
00 Context
        ↓
07 Verification Resource
        ↓
13 Confidence Scheduling
        ↓
17 Semi-Autoregressive Drafting
        ↓
23 Throughput Objective
        ↓
29 Hardware-Aware Scheduling
        ↓
37 Operating Regimes
        ↓
43 Resource Allocation
        ↓
49 Adaptive AI Infrastructure
```


## Start here

```text
draft quality
        ↓
conditional confidence
        ↓
prefix survival
        ↓
verification allocation
        ↓
expected accepted tokens
        ↓
steps per second
        ↓
throughput
```

The main idea is that throughput is not one number controlled by one subsystem. It is the product of a draft model, a confidence profile, a verification schedule, and a hardware capacity curve.


In [ ]:
from pathlib import Path
import json
import math
import zipfile
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle

# Robust paths: works from repo root, notebooks/, or standalone uploaded execution.
CWD = Path.cwd().resolve()
if (CWD / "figures").exists() and (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "figures").exists() and (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "23_throughput_objective"
SITE_IMAGES = REPO_ROOT / "site" / "2606-19348" / "images"

FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
SITE_IMAGES.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Figures:   {FIGURES}")
print(f"Results:   {RESULTS}")


## 1. Modeling the ingredients

Notebook 23 uses deliberately small numerical models so the optimization is visible. The goal is not to reproduce DeepSeek production measurements. The goal is to make the engineering dependency explicit:


a draft profile produces prefix survival, a scheduled prefix produces expected accepted tokens, and hardware load converts verification length into steps per second.


In [ ]:
def prefix_survival(confidence):
    """Cumulative survival probability a_j = prod_{i<=j} c_i."""
    confidence = np.asarray(confidence, dtype=float)
    return np.cumprod(confidence)


def expected_accepts(confidence, ell, include_bonus=True):
    """Expected accepted tokens for scheduled prefix length ell."""
    a = prefix_survival(confidence)
    base = 1.0 if include_bonus else 0.0
    return base + float(np.sum(a[:int(ell)]))


def sps_curve(batch_tokens, base=250.0, alpha=0.045, floor=28.0):
    """Illustrative hardware steps/sec curve as verification batch grows."""
    batch_tokens = np.asarray(batch_tokens, dtype=float)
    return floor + base / (1.0 + alpha * batch_tokens)


def throughput(confidence, ell, active_requests=16):
    """Expected throughput = accepted tokens * hardware steps/sec."""
    tau = expected_accepts(confidence, ell)
    B = active_requests * (1 + ell)
    sps = float(sps_curve(B))
    theta = tau * sps
    return theta, tau, B, sps

requests = {
    "chat": np.array([0.76, 0.67, 0.58, 0.49, 0.40, 0.32, 0.25, 0.20]),
    "instruction": np.array([0.84, 0.78, 0.70, 0.62, 0.53, 0.45, 0.37, 0.30]),
    "code": np.array([0.91, 0.88, 0.84, 0.78, 0.70, 0.62, 0.54, 0.46]),
    "math": np.array([0.93, 0.90, 0.86, 0.81, 0.74, 0.67, 0.58, 0.50]),
}

pd.DataFrame(requests, index=[f"c_{i}" for i in range(1,9)])


## 2. Objective overview

The throughput objective is a pipeline, not a single local decision.

```text
Draft profile → Prefix survival → Verification length → Batch tokens → Steps/sec → Throughput
```

This is the first notebook where the series becomes explicitly systems-oriented.


In [ ]:
def rounded_box(ax, xy, w, h, text, fontsize=11):
    x, y = xy
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.10",
        linewidth=1.2,
        edgecolor="black",
        facecolor="white",
    )
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize)

fig, ax = plt.subplots(figsize=(12, 3.8))
ax.set_xlim(0, 12)
ax.set_ylim(0, 3)
ax.axis("off")
ax.set_title("Throughput objective: from draft quality to serving capacity", pad=18)

labels = [
    "Draft\nquality",
    "Conditional\nconfidence",
    "Prefix\nsurvival",
    "Verification\nallocation",
    "Expected\naccepted tokens",
    "Steps/sec",
    "Throughput",
]
xs = np.linspace(0.35, 10.7, len(labels))
for i, (x, lab) in enumerate(zip(xs, labels)):
    rounded_box(ax, (x, 1.15), 1.25, 0.72, lab, fontsize=10)
    if i < len(labels) - 1:
        ax.annotate("", xy=(xs[i+1]-0.10, 1.51), xytext=(x+1.35, 1.51), arrowprops=dict(arrowstyle="->", lw=1.5))

ax.text(6, 0.28, "Notebook 23 optimizes the whole chain, not one local subsystem.", ha="center", fontsize=11)
fig.tight_layout()
path = FIGURES / "23_objective_overview.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
plt.show()
path


## 3. Expected accepted length

For scheduled prefix length \(\ell\), speculative decoding accepts a continuous prefix. If \(a_j\) is the probability that the prefix through position \(j\) survives target verification, then

\[
\tau(\ell)=1+\sum_{j=1}^{\ell}a_j.
\]

The bonus token contributes the leading \(1\). The summation contributes expected accepted draft tokens.


In [ ]:
rows = []
for name, conf in requests.items():
    a = prefix_survival(conf)
    for ell in range(0, len(conf)+1):
        theta, tau, B, sps = throughput(conf, ell, active_requests=16)
        rows.append({
            "request": name,
            "ell": ell,
            "expected_accepts_tau": tau,
            "batch_tokens_B": B,
            "steps_per_second_SPS": sps,
            "throughput_theta": theta,
            "prefix_survival_at_ell": a[ell-1] if ell > 0 else 1.0,
        })
obj = pd.DataFrame(rows)
obj.to_csv(RESULTS / "throughput_objective_table.csv", index=False)
obj.head(12)


## 4. Tradeoff surface

Adding verification tokens initially helps because the scheduler gains more expected accepted tokens. Eventually the extra verification length increases batch cost faster than it increases expected accepts. The best prefix length is therefore an operating point, not a fixed constant.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.2))
for name in requests:
    sub = obj[obj["request"] == name]
    ax.plot(sub["ell"], sub["throughput_theta"], marker="o", label=name)
    best = sub.loc[sub["throughput_theta"].idxmax()]
    ax.scatter([best["ell"]], [best["throughput_theta"]], s=80, zorder=3)
    ax.axvline(best["ell"], lw=0.8, alpha=0.22)

ax.set_xlabel("Scheduled verification length ℓ")
ax.set_ylabel("Expected throughput Θ")
ax.set_title("Throughput objective has request-dependent optima")
ax.legend(frameon=False)
fig.tight_layout()
path = FIGURES / "23_tradeoff_surface.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
plt.show()
path


## 5. Verification budget allocation

A fixed policy gives every request the same verification length. A throughput policy allocates verification where prefix survival remains high enough to justify target-model capacity.

This is the same idea introduced in Notebook 13, but now evaluated through the throughput objective.


In [ ]:
threshold = 0.35
allocation_rows = []
for name, conf in requests.items():
    a = prefix_survival(conf)
    scheduled = int(np.sum(a >= threshold))
    best = obj[obj["request"] == name].loc[obj[obj["request"] == name]["throughput_theta"].idxmax()]
    allocation_rows.append({
        "request": name,
        "threshold_length": scheduled,
        "optimal_length": int(best["ell"]),
        "optimal_theta": float(best["throughput_theta"]),
    })
alloc_df = pd.DataFrame(allocation_rows)
alloc_df.to_csv(RESULTS / "budget_allocation.csv", index=False)
alloc_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.9))
y_positions = np.arange(len(alloc_df))[::-1]
max_len = 8
for y, row in zip(y_positions, alloc_df.itertuples()):
    name = row.request
    ell = row.optimal_length
    ax.text(-0.55, y, name, ha="right", va="center", fontsize=11, fontweight="bold")
    for j in range(max_len):
        face = "black" if j < ell else "white"
        rect = Rectangle((j, y-0.28), 0.82, 0.56, facecolor=face, edgecolor="black", lw=1.0)
        ax.add_patch(rect)
    ax.text(max_len + 0.25, y, f"ℓ*={ell}", va="center", fontsize=10)

ax.set_xlim(-1.5, max_len + 1.4)
ax.set_ylim(-0.8, len(alloc_df)-0.2)
ax.set_xticks(range(max_len))
ax.set_xticklabels([str(i) for i in range(1, max_len+1)])
ax.set_yticks([])
ax.set_xlabel("Draft position")
ax.set_title("Throughput-aware verification allocation")
for spine in ax.spines.values(): spine.set_visible(False)
fig.tight_layout()
path = FIGURES / "23_budget_allocation.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
plt.show()
path


## 6. Operating regions

The optimal behavior changes with both request predictability and serving load. At low load, extra verification can be worth it. At high load, the opportunity cost of verifying low-survival suffixes increases.

The plot below is illustrative: it partitions a design space into short, medium, and long scheduled-prefix regimes.


In [ ]:
confidence_floor = np.linspace(0.2, 0.8, 61)
load = np.linspace(0.2, 1.0, 61)
Z = np.zeros((len(load), len(confidence_floor)))

for i, L in enumerate(load):
    for j, floor in enumerate(confidence_floor):
        # higher confidence supports longer prefixes; higher load shortens them
        score = 8.0 * floor * (1.08 - 0.70 * L)
        Z[i, j] = np.clip(round(score), 1, 8)

fig, ax = plt.subplots(figsize=(8, 5.4))
im = ax.imshow(Z, origin="lower", aspect="auto", extent=[confidence_floor.min(), confidence_floor.max(), load.min(), load.max()])
ax.set_xlabel("Confidence floor / draft predictability")
ax.set_ylabel("Serving load")
ax.set_title("Operating regions: optimal scheduled prefix length")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("selected ℓ")
fig.tight_layout()
path = FIGURES / "23_operating_regions.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
plt.show()
path


## 7. Optimal schedule selection

The scheduler can be viewed as a discrete optimizer. For each candidate prefix length, it estimates expected accepts, target-model batch size, hardware steps/sec, and total throughput. The selected prefix is the one that maximizes \(\Theta\).


In [ ]:
name = "code"
sub = obj[obj["request"] == name].copy()
best = sub.loc[sub["throughput_theta"].idxmax()]

fig, ax = plt.subplots(figsize=(9, 5.0))
ax.plot(sub["ell"], sub["throughput_theta"], marker="o", lw=2)
ax.axvline(best["ell"], linestyle="--", lw=1.4)
ax.scatter([best["ell"]], [best["throughput_theta"]], s=120, zorder=3)
ax.annotate(f"selected ℓ*={int(best['ell'])}", xy=(best["ell"], best["throughput_theta"]), xytext=(best["ell"]+0.6, best["throughput_theta"]-7), arrowprops=dict(arrowstyle="->"))
ax.set_xlabel("scheduled verification length ℓ")
ax.set_ylabel("expected throughput Θ")
ax.set_title("Discrete schedule selection for a structured request")
fig.tight_layout()
path = FIGURES / "23_optimal_schedule.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
plt.show()
path


## 8. Scaling with active batch size

Hardware-aware scheduling becomes more important as the number of active requests grows. The same prefix length can be reasonable at low concurrency and wasteful under high concurrency.


In [ ]:
active_grid = np.arange(4, 65, 4)
ells = np.arange(1, 9)
conf = requests["math"]
scale_rows = []
for R in active_grid:
    for ell in ells:
        theta, tau, B, sps = throughput(conf, ell, active_requests=int(R))
        scale_rows.append({"active_requests": R, "ell": ell, "theta": theta, "tau": tau, "B": B, "sps": sps})
scale_df = pd.DataFrame(scale_rows)
scale_df.to_csv(RESULTS / "batch_scaling.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5.3))
for ell in [1, 2, 4, 6, 8]:
    sub = scale_df[scale_df["ell"] == ell]
    ax.plot(sub["active_requests"], sub["theta"], marker="o", label=f"ℓ={ell}")
ax.set_xlabel("active requests R")
ax.set_ylabel("expected throughput Θ")
ax.set_title("Scheduled prefix length interacts with active batch size")
ax.legend(frameon=False, ncol=3)
fig.tight_layout()
path = FIGURES / "23_scaling.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
plt.show()
path


## 9. Architecture summary

Notebook 23 completes the first systems loop. Drafting produces candidate blocks. Confidence converts those candidates into prefix survival. The scheduler allocates target-model verification. Hardware converts batch size into steps/sec. Throughput emerges from the whole loop.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.2))
ax.set_xlim(0, 12)
ax.set_ylim(0, 4)
ax.axis("off")
ax.set_title("Confidence-scheduled serving system", pad=18)

nodes = [
    (0.7, 2.45, "Prompt"),
    (2.15, 2.45, "Semi-AR\ndrafter"),
    (3.80, 2.45, "Confidence\nprofile"),
    (5.50, 2.45, "Prefix\nsurvival"),
    (7.20, 2.45, "Verification\nscheduler"),
    (9.10, 2.45, "Target\nmodel"),
    (10.75, 2.45, "Accepted\ntokens"),
]
for i, (x, y, label) in enumerate(nodes):
    rounded_box(ax, (x, y), 1.10, 0.75, label, fontsize=9.5)
    if i < len(nodes) - 1:
        ax.annotate("", xy=(nodes[i+1][0]-0.05, y+0.38), xytext=(x+1.12, y+0.38), arrowprops=dict(arrowstyle="->", lw=1.4))

rounded_box(ax, (3.35, 0.75), 5.3, 0.75, "Throughput objective: maximize useful accepted tokens per serving capacity", fontsize=10)
ax.annotate("", xy=(6.0, 1.50), xytext=(6.0, 2.45), arrowprops=dict(arrowstyle="->", lw=1.3))
fig.tight_layout()
path = FIGURES / "23_architecture_summary.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
plt.show()
path


## Key equations

Expected accepted length:

\[
\tau(\ell)=1+\sum_{j=1}^{\ell}a_j
\]

Target-model verification batch:

\[
B=\sum_{r=1}^{R}(1+\ell_r)
\]

Hardware speed as a function of batch size:

\[
S=\mathrm{SPS}(B)
\]

Overall throughput objective:

\[
\Theta(\ell)=\tau(\ell)\,S(B(\ell))
\]

Optimal scheduled length:

\[
\ell^*=\arg\max_{\ell}\Theta(\ell)
\]


## Engineering summary

Draft quality specifies confidence. Confidence specifies schedulable prefixes. Verification allocation specifies throughput.

Notebook 23 turns this chain into an optimization problem. Notebook 29 should now ask how real hardware profiles, GPU batching, engine scheduling, and serving load change the objective.


In [ ]:
figures = sorted(str(p.relative_to(REPO_ROOT)) for p in FIGURES.glob("23_*.png"))
manifest = {
    "notebook": "23_throughput_objective.ipynb",
    "title": "23 — Throughput Objective",
    "engineering_statement": "Verification allocation specifies decoding throughput.",
    "figures": figures,
    "tables": [
        "results/23_throughput_objective/throughput_objective_table.csv",
        "results/23_throughput_objective/budget_allocation.csv",
        "results/23_throughput_objective/batch_scaling.csv",
    ],
    "next_notebook": "29_hardware_aware_scheduling.ipynb",
}
manifest_path = RESULTS / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest


In [ ]:
archive = RESULTS / "23_throughput_objective_artifacts.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in FIGURES.glob("23_*.png"):
        zf.write(path, arcname=f"figures/{path.name}")
    for path in RESULTS.glob("*.csv"):
        zf.write(path, arcname=f"results/{path.name}")
    zf.write(manifest_path, arcname="results/manifest.json")
print(f"Created {archive}")
print(f"Size: {archive.stat().st_size:,} bytes")

try:
    from IPython.display import FileLink, display
    display(FileLink(str(archive)))
except Exception:
    pass


## Next notebook

**29 — Hardware-Aware Scheduling** should ask how GPU batch behavior, verification kernels, and serving load change the optimal schedule. Notebook 23 gives the objective; Notebook 29 adds the machine.


## Download artifacts

Run the final cell to package Notebook 23 outputs for download.

The archive includes the notebook, generated figures, CSV/JSON outputs, manifests, and any site artifacts available in the current repo layout.


In [ ]:
# Package Notebook 23 artifacts for download

from pathlib import Path
from IPython.display import FileLink, display
import zipfile
import json

CWD = Path.cwd().resolve()

if (CWD / "figures").exists() or (CWD / "results").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "figures").exists() or (CWD.parent / "results").exists() or (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "23_throughput_objective"
SITE = REPO_ROOT / "site"

RESULTS.mkdir(parents=True, exist_ok=True)

manifest = {
    "notebook": "23_throughput_objective.ipynb",
    "title": "Throughput Objective",
    "engineering_statement": "Verification allocation specifies decoding throughput.",
    "primary_question": "Which verification policy maximizes end-to-end throughput?",
    "outputs": {
        "results_dir": str(RESULTS),
        "figures_dir": str(FIGURES),
    },
}
manifest_path = RESULTS / "23_throughput_objective_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

zip_path = RESULTS / "23_throughput_objective_artifacts.zip"

notebook_candidates = [
    NOTEBOOKS / "23_throughput_objective.ipynb",
    Path.cwd() / "23_throughput_objective.ipynb",
    Path.cwd() / "23_throughput_objective_full.ipynb",
    Path.cwd() / "23_throughput_objective_full_with_download.ipynb",
]
notebook_path = next((p for p in notebook_candidates if p.exists()), None)

paths_to_include = []
if notebook_path is not None:
    paths_to_include.append(notebook_path)

for item in [FIGURES, RESULTS, SITE]:
    if item.exists():
        paths_to_include.append(item)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)
        if item.is_dir():
            for path in item.rglob("*"):
                if not path.is_file():
                    continue
                if path.resolve() == zip_path.resolve():
                    continue
                try:
                    archive_name = path.relative_to(REPO_ROOT)
                except ValueError:
                    archive_name = path.name
                zf.write(path, archive_name)
        elif item.exists():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
